<a href="https://colab.research.google.com/github/yihan4326/MSSP607/blob/main/WeeklyModules/Week04/participation_week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Pandas DataFrame

This notebook uses the course file `parsed.csv`. The project structure is GitHub-friendly:

- `notebooks/Using_Pandas_DataFrame.ipynb`
- `data/parsed.csv`

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm
import statsmodels.api as se
import seaborn as sns
import datetime as dt
from sklearn.linear_model import LinearRegression
from scipy import stats
from io import StringIO
import itertools

pd.set_eng_float_format(accuracy=3, use_eng_prefix=True)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

from google.colab import userdata
import os
import subprocess

# Get GitHub token from Colab Secrets
github_token = userdata.get('Git_Hub')

# Set the GitHub repository owner and name
owner = 'yihan4326'
repository = 'MSSP607'

# Create the clone URL
clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
subprocess.run(['git', 'clone', clone_url])

# Navigate into the cloned repository
os.chdir(repository)

print(f"Changed directory to: {os.getcwd()}")
earthquakes = pd.read_csv('data/parsed.csv')

print('Data shape:', earthquakes.shape)
earthquakes.head()

Changed directory to: /content/MSSP607/MSSP607
Data shape: (9332, 27)


,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,status,time,title,tsunami,type,types,tz,updated,url,parsed_place
0,NaN,NaN,37389218,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.01,NaN,85.00,",ci37389218,",1.35,ml,...,automatic,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475395144,https://earthquake.usgs.gov/earthquakes/eventp...,California
1,NaN,NaN,37389202,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.02,NaN,79.00,",ci37389202,",1.29,ml,...,automatic,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475253925,https://earthquake.usgs.gov/earthquakes/eventp...,California
2,NaN,4.40,37389194,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.02,28.00,21.00,",ci37389194,",3.42,ml,...,automatic,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,earthquake,",dyfi,focal-mechanism,geoserve,nearby-cities,o...",-480.00,1539536756176,https://earthquake.usgs.gov/earthquakes/eventp...,California
3,NaN,NaN,37389186,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.03,NaN,39.00,",ci37389186,",0.44,ml,...,automatic,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475196167,https://earthquake.usgs.gov/earthquakes/eventp...,California
4,NaN,NaN,73096941,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.08,NaN,192.00,",nc73096941,",2.16,md,...,automatic,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,scit...",-480.00,1539477547926,https://earthquake.usgs.gov/earthquakes/eventp...,California


In [6]:
# a.the 95th percentile of earthquake magnitude in Japan using magType of "mb"
japan_mb = earthquakes[
    (earthquakes['parsed_place'].str.endswith('Japan')) &
    (earthquakes['magType'] == 'mb')]
japan_mb['mag'].quantile(0.95)

np.float64(4.9)

In [7]:
# b.the percentage of earthquakes in Indonesia that were coupled with tsunamis
indonesia = earthquakes[earthquakes['parsed_place'].str.endswith('Indonesia')]
indonesia_tsunamis = indonesia[indonesia['tsunami'] == 1]
percentage = len(indonesia_tsunamis) / len(indonesia) * 100
print(percentage)

23.12925170068027


In [11]:
# c.summary statistics for earthquakes in Nevada
nevada = earthquakes[earthquakes['parsed_place'].str.endswith('Nevada')]
nevada.describe()

,cdi,dmin,felt,gap,mag,mmi,nst,rms,sig,time,tsunami,tz,updated
count,15.00,681.00,15.00,681.00,681.00,1.00,681.00,681.00,681.00,681.00,681.00,681.00,681.00
mean,2.44,0.17,2.40,153.67,0.50,2.84,12.62,0.15,10.97,1538313805281.28,0.00,-480.00,1538401660548.00
std,0.50,0.17,4.63,68.74,0.70,NaN,9.87,0.08,19.61,596563746.20,0.00,0.00,601095107.44
min,2.00,0.00,1.00,29.14,-0.50,2.84,3.00,0.00,0.00,1537247268525.00,0.00,-480.00,1537306663881.00
25%,2.00,0.05,1.00,97.38,-0.10,2.84,6.00,0.11,0.00,1537854384162.00,0.00,-480.00,1537927505348.00
50%,2.20,0.11,1.00,149.14,0.40,2.84,10.00,0.15,2.00,1538279862199.00,0.00,-480.00,1538427614580.00
75%,2.90,0.23,1.00,199.72,0.90,2.84,16.00,0.19,12.00,1538821434704.00,0.00,-480.00,1538877980517.00
max,3.30,1.41,19.00,355.91,2.90,2.84,61.00,0.86,129.00,1539461038285.00,0.00,-480.00,1539482718719.00


In [12]:
# d. add the column of ring of fire
earthquakes['ring_of_fire'] = earthquakes['parsed_place'].str.contains(
    r'|'.join([
        'Bolivia',
        'Chile',
        'Ecuador',
        'Peru',
        'Costa Rica',
        'Guatemala',
        '^Mexico',
        'Japan',
        'Philippines',
        'Indonesia',
        'New Zealand',
        'Antarctic',
        'Canada',
        'Fiji',
        'Alaska',
        'Washington',
        'California',
        'Russia',
        'Taiwan',
        'Tonga',
        'Kermadec Islands']))

earthquakes[['parsed_place', 'ring_of_fire']].head()

,parsed_place,ring_of_fire
0,California,True
1,California,True
2,California,True
3,California,True
4,California,True


In [16]:
# e. count earthquakes inside and outside the ring of fire
total_inside = earthquakes[(earthquakes['ring_of_fire'] == True) &(earthquakes['type'] == 'earthquake')].type.count()
total_outside = earthquakes[(earthquakes['ring_of_fire'] == False) &(earthquakes['type'] == 'earthquake')].type.count()
print('The total number of earthquakes in the Ring of Fire is:', total_inside)
print('The total number of earthquakes outside the Ring of Fire is:', total_outside)

The total number of earthquakes in the Ring of Fire is: 7004
The total number of earthquakes outside the Ring of Fire is: 2077


In [10]:
# f. the tsumi count along the ring of fire
ring = earthquakes[earthquakes['ring_of_fire'] == True]
ring_tsunamis = ring[ring['tsunami'] == 1]
tsunami_count = len(ring_tsunamis)
print(tsunami_count)

45
